# 06 — Model 1 이미지 생성 (텍스트+방크기 → 후보 4장)

```
프롬프트 + 방 크기(W×D×H) → mood 자동 매칭 → 레퍼런스 4장 다양화
→ SDXL + ControlNet(canny+depth) + IP-Adapter → 후보 4장 생성
→ 마음에 안 들면 추가 텍스트로 재생성 (refine_candidates)
```

- **로컬 GPU(CUDA) 환경에서 실행** — 저장소를 clone한 로컬 환경에서 이 노트북을 직접 실행
- 방 W×D×H로 **바닥면적 기반 스케일 가드**를 프롬프트에 주입해 "5x7 원룸에 대저택 거실" 같은 비현실적 결과 방지
- 4장은 **서로 다른 레퍼런스 이미지**를 구조 가이드로 사용해 다양성 확보 (동일 사진 seed만 바꾸는 방식 아님)
- 결과물(이미지+메타데이터)은 로컬 디스크 `data/generation_cache/`에 캐싱 — 다시 실행해도 가이드 이미지는 재추출 안 해도 됨
- 생성 해상도는 SDXL 네이티브인 1024×1024 (`GENERATION_IMAGE_SIZE`, VRAM이 부족하면 낮춰서 조정)

**선행:** `04_build_library.ipynb` 실행 완료 (`mood_library/index.json` 필요)

**커널:** `.venv` Python 선택

In [ ]:
# torch, diffusers, controlnet_aux 등 Model 1 생성 스택 (커널이 .venv가 아니면 실행)
%pip install -r ../requirements-ml.txt -q


In [ ]:
# 멀티 GPU 환경에서 특정 GPU만 쓰고 싶으면, 아래 두 줄의 주석을 풀고 원하는 인덱스로 지정
# (torch import 되기 전에 실행되어야 적용됨 — 커널 재시작 후 이 셀을 가장 먼저 실행)
# import os
# os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import torch

print(f"사용 가능한 GPU 개수: {torch.cuda.device_count()}")
if torch.cuda.is_available():
    print(f"현재 활성화된 GPU 이름: {torch.cuda.get_device_name(0)}")
else:
    print("GPU를 찾지 못했습니다 — CUDA 드라이버/torch 설치를 확인하세요.")


In [ ]:
%matplotlib inline

import importlib
import json
import sys
from pathlib import Path

# notebooks/ 또는 프로젝트 루트 어디서 실행해도 루트 찾기
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "mood_pipeline").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "mood_pipeline").exists():
    raise FileNotFoundError(
        "mood_pipeline 폴더를 찾을 수 없습니다. notebooks/ 또는 프로젝트 루트에서 실행하세요."
    )

sys.path.insert(0, str(PROJECT_ROOT))

# 코드 수정 반영: config → search → generate 순서로 reload (config 캐시가 오래되면 ImportError)
import mood_pipeline.config as config_module
import mood_pipeline.generate as generate_module
import mood_pipeline.search as search_module

importlib.reload(config_module)
importlib.reload(search_module)
importlib.reload(generate_module)

from mood_pipeline.generate import generate_candidates, plot_candidates, refine_candidates, save_candidates
from mood_pipeline.search import build_mood_text_embeddings

INDEX_PATH = PROJECT_ROOT / "mood_library" / "index.json"
if not INDEX_PATH.exists():
    raise FileNotFoundError(
        f"{INDEX_PATH} 없음 → 먼저 04_build_library.ipynb 를 실행하세요."
    )

print("PROJECT_ROOT:", PROJECT_ROOT)
print("GENERATION_CACHE_DIR:", config_module.GENERATION_CACHE_DIR)
print("GENERATION_OUTPUT_DIR:", config_module.GENERATION_OUTPUT_DIR)
build_mood_text_embeddings()


## 첫 생성 — 프롬프트 + 방 크기 입력

- `width_m`/`depth_m`: 방 가로·세로(㎡ 계산용), `height_m`: 층고
- `num_candidates`: 후보 장수 (기본 4. VRAM이 부족한 GPU에서 죽으면 낮춰서 사용)
- 최초 파이프라인 로드(SDXL+ControlNet(canny+depth)+IP-Adapter 다운로드)는 몇 분 걸릴 수 있음, 이후 호출부터는 캐시된 파이프라인 재사용

In [ ]:
# ↓ 프롬프트/방 크기만 바꿔서 실행
prompt = "미니멀하고 따뜻한 우드톤의 거실을 원해. 소파와 티테이블, TV가 있는 구조로 배치해줘."
width_m, depth_m, height_m = 5.0, 7.0, 2.4

result = generate_candidates(
    prompt,
    width_m=width_m,
    depth_m=depth_m,
    height_m=height_m,
    num_candidates=4,
)

print("선택된 무드:", result["mood_id"])
print("무드 확신도:", result["mood_confidence"], "(1.0=확실한 매칭, 낮으면 프리셋/레퍼런스 영향력을 줄여서 적용)")
if result["mood_scores"]:
    print("무드 후보 점수:", [(m["mood_id"], round(m["score"], 4)) for m in result["mood_scores"]])
print("방 스케일 가드:", result["room"])
print("최종 프롬프트:", result["positive_prompt"])

plot_candidates(result, prompt=result["prompt_en"])
saved = save_candidates(result)
print("저장 위치:", saved["meta_path"])


## 마음에 드는 게 없으면 → 추가 텍스트로 재생성

- 같은 무드 안에서, **이전에 썼던 레퍼런스 이미지는 제외**하고 새 4장을 뽑음
- `feedback`에 추가로 원하는 조건만 적으면 됨 (원래 프롬프트와 자동으로 합쳐짐)

In [ ]:
# ↓ 추가 요청 사항만 바꿔서 실행 (여러 번 반복 가능)
feedback = "좀 더 밝고 채광이 좋은 느낌으로, 식물도 하나 넣어줘"

refined = refine_candidates(result, feedback, num_candidates=4)

print("최종 프롬프트:", refined["positive_prompt"])
plot_candidates(refined, prompt=refined["prompt_en"])
saved_refined = save_candidates(refined)
print("저장 위치:", saved_refined["meta_path"])

# 다음 재생성에서도 계속 새로운 레퍼런스가 나오도록 result를 최신 결과로 갱신
result = refined


## 최종 선택 — model2로 넘길 이미지

`chosen_index`에 사용자가 고른 후보 번호(0부터, `num_candidates` 개수 미만)를 넣으면 해당 이미지 경로 + 메타데이터를 정리해줌.
이 결과(`final_selection`)가 diagram상 **model2의 입력 1번(model1에서 선택한 이미지)**이 됨.

In [ ]:
chosen_index = 0  # ↓ 사용자가 고른 후보 번호로 변경

chosen = saved_refined["candidates"][chosen_index]
final_selection = {
    "image_path": chosen["image_path"],
    "mood_id": refined["mood_id"],
    "room": refined["room"],
    "prompt_used": refined["positive_prompt"],
    "reference_path": chosen["reference_path"],
}
print(json.dumps(final_selection, ensure_ascii=False, indent=2))
